# 🛟 Lekcja 2: Kto dostał miejsce w szalupie? Wizualizacja zależności i potęga `.groupby()`

W Lekcji 1 odkryliśmy, że katastrofę Titanica przeżyło ogółem **38.4% pasażerów** (342 z 891 osób).

Ale czy miejsca w szalupach ratunkowych były rozdzielane losowo jak na loterii? Oczywiście, że nie! Podczas ewakuacji obowiązywała słynna morska zasada **„Kobiety i dzieci przodem!”**, a ogromne znaczenie miało też to, czy kabina pasażera znajdowała się wysoko przy szalupach (**1. klasa**), czy głęboko pod pokładem (**3. klasa**).

W tej lekcji nauczymy się:
1. Jak za pomocą jednej z najważniejszych metod w `pandas` — **`.groupby()` (Grupuj według...)** — sprawdzić szanse przeżycia dla dowolnej grupy pasażerów.
2. Jak zwizualizować na wykresach wpływ **płci (`Sex`)**, **klasy biletu (`Pclass`)** oraz **wieku (`Age`)** na przeżycie katastrofy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Wczytujemy dane z folderu ../data/
train = pd.read_csv('../data/train.csv')

# 2. Obliczamy procent ocalałych według płci (Sex) oraz według klasy biletu (Pclass)
przezycie_plec = (train.groupby('Sex')['Survived'].mean() * 100).round(1)
przezycie_klasa = (train.groupby('Pclass')['Survived'].mean() * 100).round(1)

print('--- Odsetek ocalałych według płci (Sex) [%] ---')
print(przezycie_plec)

print('\n--- Odsetek ocalałych według klasy biletu (Pclass) [%] ---')
print(przezycie_klasa)

# 3. WIZUALIZACJA: Dwa wykresy słupkowe obok siebie
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Wykres lewy: Płeć a przeżycie
etykiety_plci = ['Kobiety (female)', 'Mężczyźni (male)']
wartosci_plci = [przezycie_plec['female'], przezycie_plec['male']]
ax1.bar(etykiety_plci, wartosci_plci, color=['orchid', 'steelblue'], edgecolor='black', width=0.45)
ax1.axhline(38.4, color='red', linestyle='--', linewidth=1.5, label='Średnia dla całego statku (38.4%)')
ax1.set_title('1. Szansa przeżycia według płci (Sex)')
ax1.set_ylabel('Procent ocalałych (%)')
ax1.set_ylim(0, 100)
for i, v in enumerate(wartosci_plci):
    ax1.text(i, v + 2.5, f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax1.legend()
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: Klasa biletu a przeżycie
etykiety_klas = ['1. Klasa\n(Najbogatsi)', '2. Klasa\n(Średnia)', '3. Klasa\n(Emigranci)']
wartosci_klas = [przezycie_klasa[1], przezycie_klasa[2], przezycie_klasa[3]]
ax2.bar(etykiety_klas, wartosci_klas, color=['gold', 'skyblue', 'lightcoral'], edgecolor='black', width=0.5)
ax2.axhline(38.4, color='red', linestyle='--', linewidth=1.5, label='Średnia dla całego statku (38.4%)')
ax2.set_title('2. Szansa przeżycia według klasy biletu (Pclass)')
ax2.set_ylabel('Procent ocalałych (%)')
ax2.set_ylim(0, 100)
for i, v in enumerate(wartosci_klas):
    ax2.text(i, v + 2.5, f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 🔬 Co dzieje się pod maską w `train.groupby('Sex')['Survived'].mean()`? (Zasada *Rozdziel $\rightarrow$ Policz $\rightarrow$ Połącz*)

Metoda **`.groupby()`** to odpowiednik tabel przestawnych (*Pivot Tables*) z Excela. Kiedy piszesz `train.groupby('Sex')['Survived'].mean()`, komputer wykonuje w pamięci 3 proste kroki:

| Krok pod maską | Co robi komputer? | Wynik pośredni |
| :--- | :--- | :--- |
| **1. ROZDZIEL (`.groupby('Sex')`)** | Rozdziela 891 pasażerów do dwóch osobnych koszyków według płci. | • Koszyk `'female'`: 314 kobiet<br>• Koszyk `'male'`: 577 mężczyzn |
| **2. WYBIERZ (`['Survived']`)** | Z każdego koszyka wyciąga tylko kolumnę z informacją o przeżyciu (`0` lub `1`). | Dwie listy pełne zer i jedynek |
| **3. POLICZ I POŁĄCZ (`.mean() * 100`)** | W każdym koszyku osobno liczy średnią z zer i jedynek (czyli ułamek ocalałych) i mnoży przez 100%. | • `'female'` $\rightarrow$ **`74.2%`** (prawie 3 na 4 kobiety przeżyły!)<br>• `'male'` $\rightarrow$ **`18.9%`** (mniej niż 1 na 5 mężczyzn przeżył!) |

---

#### 💡 Co odkryliśmy na powyższych wykresach?
1. **Płeć (`Sex`) to najsilniejsza pojedyncza cecha w całym zbiorze Titanica!**  
   Różnica między **`74.2%`** (kobiety) a **`18.9%`** (mężczyźni) jest gigantyczna. Gdyby nasz model nie wiedział o pasażerze zupełnie nic poza jego płcią i obstawiał prostą regułę: *„Każda kobieta przeżyła (`1`), każdy mężczyzna zginął (`0`)”*, już na starcie miałby prawie **78% trafnych odpowiedzi**! (Dokładnie tak powstał przykładowy plik `gender_submission.csv` w naszym folderze `../data/`!).
2. **Klasa biletu (`Pclass`) tworzy wyraźne schody ratunku:**  
   W **1. klasie** uratowało się aż **`63.0%`** pasażerów, w **2. klasie `47.3%`**, a w **3. klasie** zaledwie **`24.2%`**.

In [ ]:
# Sprawdźmy teraz dwie rzeczy naraz:
# A) Jak wiek (Age podzielony na przedziały wiekowe pd.cut) wpływał na przeżycie?
# B) Jak połączona Płeć ORAZ Klasa biletu (.groupby(['Pclass', 'Sex'])) zmieniały szanse ratunku?

# 1. Dzielimy wiek na 5 czytelnych przedziałów wiekowych za pomocą funkcji pd.cut()
granice_wieku = [0, 12, 18, 35, 55, 100]
nazwy_grup = ['1. Dzieci\n(0-12 lat)', '2. Młodzież\n(13-18 lat)', '3. Dorośli\n(19-35 lat)', '4. Dojrzali\n(36-55 lat)', '5. Seniorzy\n(55+ lat)']
train['Grupa_Wiekowa'] = pd.cut(train['Age'], bins=granice_wieku, labels=nazwy_grup)

przezycie_wiek = (train.groupby('Grupa_Wiekowa', observed=False)['Survived'].mean() * 100).round(1)

# 2. Grupujemy jednocześnie po DWOCH kolumnach: Klasie biletu i Płci!
przezycie_klasa_plec = (train.groupby(['Pclass', 'Sex'])['Survived'].mean() * 100).round(1)
print('--- Szansa przeżycia według Klasy ORAZ Płci [%] ---')
print(przezycie_klasa_plec)

# 3. WIZUALIZACJA: Po lewej grupy wiekowe, po prawej połączenie Płci i Klasy biletu!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Wykres lewy: Grupy wiekowe
kolory_wieku = ['mediumseagreen', 'skyblue', 'steelblue', 'steelblue', 'salmon']
ax1.bar(przezycie_wiek.index, przezycie_wiek.values, color=kolory_wieku, edgecolor='black', width=0.55)
ax1.axhline(38.4, color='red', linestyle='--', linewidth=1.5, label='Średnia statku (38.4%)')
ax1.set_title('1. Szansa przeżycia w grupach wiekowych (Dzieci przodem!)')
ax1.set_ylabel('Procent ocalałych (%)')
ax1.set_ylim(0, 100)
for i, v in enumerate(przezycie_wiek.values):
    ax1.text(i, v + 2.5, f'{v}%', ha='center', fontweight='bold')
ax1.legend()
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: Kobiety vs Mężczyźni w każdej klasie biletu (1, 2, 3)
klasy = ['1. Klasa', '2. Klasa', '3. Klasa']
kobiety_wg_klasy = [przezycie_klasa_plec[(1, 'female')], przezycie_klasa_plec[(2, 'female')], przezycie_klasa_plec[(3, 'female')]]
mezczyzni_wg_klasy = [przezycie_klasa_plec[(1, 'male')], przezycie_klasa_plec[(2, 'male')], przezycie_klasa_plec[(3, 'male')]]

x = np.arange(len(klasy))
szer = 0.35
ax2.bar(x - szer/2, kobiety_wg_klasy, width=szer, color='orchid', edgecolor='black', label='Kobiety (female)')
ax2.bar(x + szer/2, mezczyzni_wg_klasy, width=szer, color='steelblue', edgecolor='black', label='Mężczyźni (male)')
ax2.set_title('2. Szansa przeżycia: Płeć + Klasa biletu razem!')
ax2.set_xticks(x)
ax2.set_xticklabels(klasy)
ax2.set_ylabel('Procent ocalałych (%)')
ax2.set_ylim(0, 110)
for i, v in enumerate(kobiety_wg_klasy):
    ax2.text(i - szer/2, v + 2, f'{v}%', ha='center', fontweight='bold', fontsize=10)
for i, v in enumerate(mezczyzni_wg_klasy):
    ax2.text(i + szer/2, v + 2, f'{v}%', ha='center', fontweight='bold', fontsize=10)
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 📊 Spójrz na prawy wykres powyżej — tak właśnie rodzi się drzewo decyzyjne!

Kiedy połączyliśmy ze sobą dwie cechy: **`Pclass` ORAZ `Sex`** (`train.groupby(['Pclass', 'Sex'])`), zobaczyliśmy obraz katastrofy z niezwykłą ostrością:
* **Kobiety z 1. i 2. klasy** miały niemal gwarantowany ratunek: przeżyło aż **`96.8%`** (1. klasa) oraz **`92.1%`** (2. klasa)!
* **Kobiety z 3. klasy** miały dokładnie **`50.0%`** szans (połowa przeżyła, połowa zginęła — tu o życiu decydowały dodatkowe cechy, np. wielkość rodziny czy cena biletu).
* **Mężczyźni z 1. klasy** przeżyli w **`36.9%`**, natomiast mężczyźni z 2. i 3. klasy mieli tylko **`15.7%`** oraz **`13.5%`** szans na ratunek.

#### Nowa funkcja w kodzie: `pd.cut(train['Age'], bins=[0, 12, 18, 35, 55, 100], labels=[...])`
* Słowo `cut` oznacza **„potnij na kawałki”**. Zamiast analizować każdy wiek osobno (21 lat, 22 lata, 23 lata...), pocięliśmy oś wieku w punktach `0, 12, 18, 35, 55, 100` na 5 życiowych szufladek. Na lewym wykresie od razu widać, że **Dzieci (0–12 lat)** miały najwyższą przeżywalność ze wszystkich grup wiekowych (**`58.0%`**)!

In [ ]:
# 🎯 ĆWICZENIE 1: Czy port zaokrętowania ('Embarked': C = Cherbourg, Q = Queenstown, S = Southampton)
# miał wpływ na szansę przeżycia?
#
# 1. Użyj metody .groupby('Embarked')['Survived'].mean() * 100 na tabeli `train`
#    i wyświetl wynik za pomocą print().
# 2. Zastanów się (lub sprawdź za pomocą train.groupby('Embarked')['Pclass'].mean()):
#    dlaczego pasażerowie wsiadający we francuskim porcie 'C' (Cherbourg) mieli wyższą przeżywalność (55.4%)?

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Filtrowanie łączone — małe dzieci w 1. i 2. klasie!
#
# Sprawdźmy, jaki procent dzieci poniżej 10 roku życia (Age < 10) płynących w 1. lub 2. klasie (Pclass <= 2)
# uratował się z katastrofy!
#
# 1. Stwórz zmienną `dzieci_1_2`, filtrując `train` dwoma warunkami naraz:
#    (train['Age'] < 10) & (train['Pclass'] <= 2)
# 2. Wypisz liczbę takich dzieci (len(dzieci_1_2)) oraz procent ocalałych (dzieci_1_2['Survived'].mean() * 100)!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Zasada działania `.groupby()`:** Konstrukcja `train.groupby('Kolumna')['Survived'].mean() * 100` dzieli pasażerów na podgrupy i w każdej z nich oblicza dokładny procent ocalałych.
2. **Trzy główne czynniki ratunku:** Największe szanse przeżycia miały **kobiety (`74.2%` vs `18.9%` u mężczyzn)**, **pasażerowie 1. klasy (`63.0%` vs `24.2%` w 3. klasie)** oraz **dzieci do 12. roku życia (`58.0%`)**.
3. **Łączenie cech (`['Pclass', 'Sex']`):** Połączenie płci z klasą biletu daje ogromną siłę predykcyjną — w 1. i 2. klasie uratowało się ponad `92–96%` kobiet, podczas gdy wśród mężczyzn z 2. i 3. klasy przeżyło zaledwie `13.5–15.7%`.